# Stage 2 — Dark Store Placement: K-Means Clustering

**Project:** Dark Store Placement + Integrated Forward & Reverse Logistics  
**Author:** Sneha

**Depends on:** `data/master_df_v2.parquet` — produced by `notebooks/02_demand_and_baseline.ipynb`

**Outputs:**
- `data/dark_store_candidates.csv` — K-Means centroids for every K tried
- `data/dark_stores_final.csv` — Chosen K with coverage + capacity KPIs
- `data/master_df_v2.parquet` — Updated with `dark_store_id` column
- `outputs/elbow_silhouette.png`
- `outputs/kmeans_cluster_map.png`
- `outputs/zone_demand_bars.png`

---

All heavy logic lives in `src/clustering.py`. This notebook calls it, then focuses on visualisation and interpretation.

**Why `master_df_v2` and not `master_df`?**  
`demand_baseline.py` adds `demand_per_zip` to every row — the number of orders from each zip code. We pass this as `sample_weight` to K-Means so high-demand areas attract dark store centroids toward them.

In [ ]:
# ── Path setup ── run this first so Python can find src/clustering.py
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.clustering import (
    load_sp_data,
    build_zip_level_coords,
    run_kmeans,
    pick_optimal_k,
    run_p_median,
    build_p_median_locations_df,
    save_p_median_outputs,
    assign_voronoi,
    haversine_km,
    compute_coverage,
    build_dark_stores_df,
    save_outputs,
)

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

DATA_DIR   = "../data"
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Imports OK.")

---
## 1. Load data

We load `master_df_v2.parquet` which already has:
- `demand_per_zip` — total orders from each zip code (used as K-Means weight)
- `cust_seller_dist_km` — baseline distance metric

Both were added by `demand_baseline.py`.

In [ ]:
df = load_sp_data(f"{DATA_DIR}/master_df_v2.parquet")

print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print()
df[["customer_lat", "customer_lon", "order_value",
    "demand_per_zip", "is_return"]].describe().round(3)

In [ ]:
# Load the baseline KPIs produced by demand_baseline — we'll use them in the
# comparison table at the end
with open(f"{DATA_DIR}/baseline_kpis.json") as f:
    baseline_kpis = json.load(f)

print("Baseline KPIs (naive — no dark stores):")
print(f"  Mean customer→seller distance : {baseline_kpis['mean_cust_seller_dist_km']} km")
print(f"  Median customer→seller        : {baseline_kpis['median_cust_seller_dist_km']} km")
print(f"  Mean delivery days            : {baseline_kpis['mean_delivery_days']}")
print(f"  Return rate                   : {baseline_kpis['return_rate_pct']}%")
print(f"  Dark store coverage (before)  : {baseline_kpis['dark_store_coverage_pct']}%")

---
## 2. Sanity check — plot raw customer locations

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(df["customer_lon"], df["customer_lat"],
           s=1, alpha=0.15, color="steelblue")
ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title(
    f"São Paulo customers — {len(df):,} orders, "
    f"{df['customer_zip_code_prefix'].nunique():,} zip codes",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sp_customer_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/sp_customer_scatter.png")

---
## 3. Aggregate to zip-code level

K-Means gets one point per zip code, not one point per order.  
Each point is weighted by `demand_per_zip` (from `demand_baseline.py`) so high-demand areas attract centroids.

In [ ]:
zip_df = build_zip_level_coords(df)

coords  = zip_df[["lat", "lon"]].values     # shape (n_zips, 2)
weights = zip_df["demand_weight"].values     # shape (n_zips,)

print(f"coords  shape: {coords.shape}   ← (n_zip_codes, 2)")
print(f"weights shape: {weights.shape}  ← demand_per_zip per zip")
print()
zip_df.sort_values("demand_weight", ascending=False).head(10)

---
## 4. K-Means sweep — try K = 3 to 12

In [ ]:
K_RANGE = range(3, 13)

print(f"Running K-Means for K = {K_RANGE.start} … {K_RANGE.stop - 1}")
print("-" * 50)
kmeans_results = run_kmeans(coords, weights, K_RANGE)
print("-" * 50)
print("Done.")

---
## 5. Choose optimal K — elbow curve + silhouette score

In [ ]:
k_list      = list(K_RANGE)
inertias    = [kmeans_results[k]["inertia"]    for k in k_list]
silhouettes = [kmeans_results[k]["silhouette"] for k in k_list]
optimal_k   = pick_optimal_k(kmeans_results)

# Print summary table
summary_df = pd.DataFrame({
    "K":          k_list,
    "Inertia":    [round(kmeans_results[k]["inertia"], 2)    for k in k_list],
    "Silhouette": [round(kmeans_results[k]["silhouette"], 4) for k in k_list],
})
summary_df["Selected"] = summary_df["K"].apply(lambda k: "★" if k == optimal_k else "")
print(summary_df.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Elbow curve
ax1.plot(k_list, inertias, marker="o", color="steelblue", linewidth=2)
ax1.axvline(optimal_k, color="crimson", linestyle="--", linewidth=1.5,
            label=f"Chosen K = {optimal_k}")
ax1.set_xlabel("Number of dark stores (K)", fontsize=12)
ax1.set_ylabel("Inertia (within-cluster sum of squares)", fontsize=12)
ax1.set_title("Elbow Curve", fontsize=13, fontweight="bold")
ax1.set_xticks(k_list)
ax1.legend(fontsize=11)

# Silhouette bars
bar_cols = ["crimson" if k == optimal_k else "steelblue" for k in k_list]
ax2.bar(k_list, silhouettes, color=bar_cols, edgecolor="white")
ax2.set_xlabel("Number of dark stores (K)", fontsize=12)
ax2.set_ylabel("Silhouette score (higher = better)", fontsize=12)
ax2.set_title("Silhouette Score by K", fontsize=13, fontweight="bold")
ax2.set_xticks(k_list)
ax2.bar([], [], color="crimson", label=f"Best K = {optimal_k}")
ax2.legend(fontsize=11)

plt.suptitle(
    f"Choosing the optimal number of dark stores — best K = {optimal_k}",
    fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/elbow_silhouette.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → outputs/elbow_silhouette.png")
print(f"Optimal K = {optimal_k}  |  Silhouette = {kmeans_results[optimal_k]['silhouette']:.4f}")

---
## 6. Voronoi zone assignment

Assign every customer row in `master_df_v2` to their nearest dark store centroid.  
This creates the `dark_store_id` column that Pritam's VRP pipeline needs.

In [ ]:
centroids = kmeans_results[optimal_k]["centroids"]   # shape (K, 2)
customer_coords = df[["customer_lat", "customer_lon"]].values

df["dark_store_id"] = assign_voronoi(customer_coords, centroids)

print("Zone assignment complete.")
print()
print("Orders per zone:")
print(df.groupby("dark_store_id").size().rename("n_orders").to_frame().to_string())

---
## 7. Coverage check

Project KPI: **> 70%** of customers within **5 km** of their dark store.

In [ ]:
coverage_overall = compute_coverage(customer_coords, centroids, radius_km=5.0)

print(f"Overall 5-km coverage : {coverage_overall * 100:.1f}%")
print(f"Target                : > 70%")
print(f"Status                : {'✓ MET' if coverage_overall >= 0.70 else '✗ NOT MET — consider increasing K'}")
print()
print("Per-zone coverage:")
zone_ids = assign_voronoi(customer_coords, centroids)
for z in range(optimal_k):
    mask = zone_ids == z
    if mask.sum() == 0:
        continue
    cov = compute_coverage(customer_coords[mask], centroids[z:z+1], radius_km=5.0)
    print(f"  Zone {z}: {cov * 100:.1f}%  ({mask.sum():,} orders)")

In [ ]:
# Find minimum K that achieves >= 70% coverage target
print("Coverage by K:")
print("-" * 35)
coverage_by_k = {}
for k, res in kmeans_results.items():
    c = compute_coverage(customer_coords, res["centroids"], radius_km=5.0)
    coverage_by_k[k] = round(c * 100, 1)
    print(f"  K={k:2d}  coverage={c*100:.1f}%")

viable = {k: v for k, v in coverage_by_k.items() if v >= 70.0}
if viable:
    optimal_k = min(viable)
    print(f"\nMinimum K hitting 70% target: K={optimal_k} ({coverage_by_k[optimal_k]}%)")
else:
    optimal_k = max(coverage_by_k, key=coverage_by_k.get)
    print(f"\nNo K in range hits 70% — best available: K={optimal_k} ({coverage_by_k[optimal_k]}%)")
    print("Consider: increase k_range upper bound, or tighten metro filter.")

# Re-assign with coverage-optimal K
centroids = kmeans_results[optimal_k]["centroids"]
df["dark_store_id"] = assign_voronoi(customer_coords, centroids)
coverage_overall = compute_coverage(customer_coords, centroids, radius_km=5.0)

---
## 8. Visualise — cluster map

In [ ]:
ZONE_COLOURS = [
    "#E63946", "#457B9D", "#2A9D8F", "#E9C46A", "#264653",
    "#F4A261", "#A8DADC", "#6D6875", "#B5838D", "#E07A5F",
    "#8338EC", "#FB5607",
]

# Sample 10 k rows for plotting speed
sample_idx = np.random.choice(len(df), size=min(10_000, len(df)), replace=False)
sample_df  = df.iloc[sample_idx]

fig, ax = plt.subplots(figsize=(9, 9))

for z in range(optimal_k):
    mask = sample_df["dark_store_id"] == z
    ax.scatter(
        sample_df.loc[mask, "customer_lon"],
        sample_df.loc[mask, "customer_lat"],
        s=4, alpha=0.35, color=ZONE_COLOURS[z % len(ZONE_COLOURS)],
        label=f"Zone {z}"
    )

# Dark store stars
ax.scatter(
    centroids[:, 1], centroids[:, 0],
    s=350, marker="*", color="black", zorder=6, label="Dark store"
)
for z, (lat, lon) in enumerate(centroids):
    ax.annotate(f" DS-{z}", xy=(lon, lat), fontsize=9,
                fontweight="bold", color="black", zorder=7)

ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
ax.set_title(
    f"K-Means Dark Store Placement — São Paulo (K = {optimal_k})\n"
    f"★ = dark store  ·  colours = service zones  ·  "
    f"5-km coverage = {coverage_overall * 100:.1f}%",
    fontsize=12, fontweight="bold"
)
ax.legend(loc="lower right", fontsize=8, ncol=2, framealpha=0.9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/kmeans_cluster_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/kmeans_cluster_map.png")

## 9. p-Median MILP validation

We use the K-Means centroids as candidate facility locations and let the p-Median MILP pick p = optimal_k of them to open.  
**Agreement criterion:** all p-Median facilities match a K-Means centroid exactly (they're the same candidate set), so we check that the MILP opens the same K centroids K-Means identified — confirming both methods agree.

We sample 100 zip codes to keep CBC runtime under 60 seconds.

In [ ]:
try:
    import pulp
    PULP_OK = True
except ImportError:
    PULP_OK = False
    print("PuLP not installed — skipping p-Median validation.")
    print("Install with: pip install pulp")

if PULP_OK:
    N_SAMPLE = min(100, len(zip_df))
    zip_sample = zip_df.sample(N_SAMPLE, random_state=42)
    cust_pm    = zip_sample[["lat", "lon"]].values
    demand_pm  = zip_sample["demand_weight"].values

    # Build Haversine distance matrix: (100 zips) × (K centroids)
    dist_pm = np.zeros((N_SAMPLE, optimal_k))
    for i in range(N_SAMPLE):
        for j in range(optimal_k):
            dist_pm[i, j] = haversine_km(
                cust_pm[i, 0], cust_pm[i, 1],
                centroids[j, 0], centroids[j, 1],
            )

    print(f"Running p-Median MILP (n_customers={N_SAMPLE}, n_candidates={optimal_k}, p={optimal_k})...")
    opened = run_p_median(dist_pm, demand_pm, p=optimal_k)

    print()
    print("p-Median opened facility indices (= K-Means centroids):", opened)
    if sorted(opened) == list(range(optimal_k)):
        print("✓ p-Median confirms all K-Means centroids — high confidence in placement.")
    else:
        print("Note: p-Median selected a subset of centroids. Review with team.")
        print("      This can happen when some zones are very close together.")

In [ ]:
# Build p-Median locations DataFrame and compute weighted distances
if PULP_OK:
    p_median_df = build_p_median_locations_df(centroids, opened)
    print("p-Median facility locations:")
    print(p_median_df.to_string(index=False))
    print()

    # Weighted distance: K-Means
    # For each sampled zip, find nearest K-Means centroid and compute demand-weighted distance
    km_assignments = assign_voronoi(cust_pm, centroids)
    km_dists = haversine_km(
        cust_pm[:, 0], cust_pm[:, 1],
        centroids[km_assignments, 0], centroids[km_assignments, 1],
    )
    km_weighted_dist = float((km_dists * demand_pm).sum())

    # Weighted distance: p-Median
    # Each zip is assigned to its nearest opened p-Median facility
    pm_coords = p_median_df[["lat", "lon"]].values
    pm_assignments = assign_voronoi(cust_pm, pm_coords)
    pm_dists = haversine_km(
        cust_pm[:, 0], cust_pm[:, 1],
        pm_coords[pm_assignments, 0], pm_coords[pm_assignments, 1],
    )
    pm_weighted_dist = float((pm_dists * demand_pm).sum())

    print(f"K-Means total weighted distance : {km_weighted_dist:,.1f} km·orders")
    print(f"p-Median total weighted distance: {pm_weighted_dist:,.1f} km·orders")
    improvement_pct = (km_weighted_dist - pm_weighted_dist) / km_weighted_dist * 100
    print(f"p-Median improvement            : {improvement_pct:.2f}%")
    print()
    print("Note: both methods use the same K-Means centroids as candidate set,")
    print("so improvement reflects the MILP's optimal assignment, not new locations.")

In [ ]:
# Build and display the comparison table
if PULP_OK:
    comparison_df = pd.DataFrame([
        {
            "method":                  "K-Means",
            "n_facilities":            optimal_k,
            "n_sample_zips":           N_SAMPLE,
            "total_weighted_dist_km":  round(km_weighted_dist, 2),
            "mean_weighted_dist_km":   round(km_weighted_dist / demand_pm.sum(), 4),
            "pct_improvement_vs_kmeans": 0.0,
        },
        {
            "method":                  "p-Median MILP",
            "n_facilities":            optimal_k,
            "n_sample_zips":           N_SAMPLE,
            "total_weighted_dist_km":  round(pm_weighted_dist, 2),
            "mean_weighted_dist_km":   round(pm_weighted_dist / demand_pm.sum(), 4),
            "pct_improvement_vs_kmeans": round(improvement_pct, 2),
        },
    ])
    print("K-Means vs p-Median comparison:")
    print(comparison_df.to_string(index=False))

In [ ]:
# Save p-Median outputs
if PULP_OK:
    save_p_median_outputs(p_median_df, comparison_df, out_dir=DATA_DIR)
    print()
    print("Files written:")
    print("  data/p_median_locations.csv            — opened facility lat/lon")
    print("  data/KMeans_vs_pMedian_comparison.csv  — side-by-side weighted distance comparison")

## 10. Dark store summary table + demand distribution

In [ ]:
dark_stores_df = build_dark_stores_df(centroids, df)
print("Dark stores final table:")
dark_stores_df

In [ ]:
zone_summary = (
    df.groupby("dark_store_id")
    .agg(n_orders=("order_id", "count"),
         total_value=("order_value", "sum"))
    .reset_index()
)
bar_cols = [ZONE_COLOURS[z % len(ZONE_COLOURS)] for z in zone_summary["dark_store_id"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.bar(zone_summary["dark_store_id"], zone_summary["n_orders"],
        color=bar_cols, edgecolor="white")
ax1.set_xlabel("Dark store zone", fontsize=12)
ax1.set_ylabel("Orders", fontsize=12)
ax1.set_title("Orders per zone", fontsize=13, fontweight="bold")
ax1.set_xticks(zone_summary["dark_store_id"])

ax2.bar(zone_summary["dark_store_id"], zone_summary["total_value"] / 1e3,
        color=bar_cols, edgecolor="white")
ax2.set_xlabel("Dark store zone", fontsize=12)
ax2.set_ylabel("Order value (R$ thousands)", fontsize=12)
ax2.set_title("Order value per zone", fontsize=13, fontweight="bold")
ax2.set_xticks(zone_summary["dark_store_id"])

plt.suptitle("Demand distribution across dark store zones", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/zone_demand_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/zone_demand_bars.png")

---
## 11. Before vs After comparison

Compare baseline KPIs (no dark stores) with what clustering gives us.

In [ ]:
# After: mean customer → dark store distance
zone_ids = assign_voronoi(customer_coords, centroids)
after_dists = haversine_km(
    customer_coords[:, 0], customer_coords[:, 1],
    centroids[zone_ids, 0], centroids[zone_ids, 1],
)

comparison = {
    "Metric":                           ["Mean dist to fulfilment point (km)",
                                         "Dark store coverage (%)",
                                         "Number of dark stores"],
    "Baseline (seller→customer, naive)": [baseline_kpis["mean_cust_seller_dist_km"],
                                          baseline_kpis["dark_store_coverage_pct"],
                                          baseline_kpis["num_dark_stores"]],
    "After K-Means clustering":         [round(float(after_dists.mean()), 2),
                                          round(coverage_overall * 100, 1),
                                          optimal_k],
}
comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))
print()
improvement = (
    (baseline_kpis["mean_cust_seller_dist_km"] - float(after_dists.mean()))
    / baseline_kpis["mean_cust_seller_dist_km"] * 100
)
print(f"Distance reduction: {improvement:.1f}%")

---
## 12. Save all outputs

In [ ]:
save_outputs(kmeans_results, dark_stores_df, df, out_dir=DATA_DIR)
print()
print("Files written:")
print(f"  data/dark_store_candidates.csv    — centroids for all K = 3…12")
print(f"  data/dark_stores_final.csv        — K={optimal_k} with KPI table")
print(f"  data/master_df_v2.parquet         — with dark_store_id column added")

---
## Deliverables checklist

| Output | Status |
|--------|--------|
| `data/dark_store_candidates.csv` | ✅ |
| `data/dark_stores_final.csv` | ✅ |
| `data/master_df_v2.parquet` (with `dark_store_id`) | ✅ |
| `outputs/elbow_silhouette.png` | ✅ |
| `outputs/kmeans_cluster_map.png` | ✅ |
| `outputs/zone_demand_bars.png` | ✅ |

**Handoffs:**
- `dark_stores_final.csv` → **Pritam** (forward VRP: one subproblem per zone)
- `master_df_v2.parquet` (with `dark_store_id`) → **Vybhav** (reverse VRP + return classifier)
- Zone demand data → **Anurag** (Prophet demand forecasting per zone)
- Maps → **Varsha** (Folium interactive map)

In [ ]:
df['customer_city'].value_counts().head(10)